# Study 884 — Convexity Barbell 🏋️

**Does a duration-matched SHY+TLT barbell out-earn the IEF bullet on its extra convexity?**

Textbook fixed income (Fabozzi; Ilmanen): a **barbell** (short + long ends) weighted to the
same **duration** as a **bullet** (the belly) carries more **convexity**, so the second-order
term `+½·C·(Δy)²` should make it out-earn the bullet whenever yields move a lot. We rebuild
it from three iShares Treasury ETFs + a cash leg —
**bullet = IEF (7-10y)**, **barbell = 0.605·SHY (1-3y) + 0.395·TLT (20y+)**,
duration-matched each day — over 2010-01-04 → 2026-06-30 (4,147 rows).

*Numbers below are the frozen headline (`docs/results.md`); the live cells run the fast
synthetic control. Fingerprint `32356eb6aefe`.*


## 1. The idea in one picture

Two Treasury books with the **same duration** (same first-order rate exposure): the **bullet** just holds the belly (IEF); the **barbell** holds the short and long ends (SHY + TLT) weighted to match. Because convexity grows with the square of maturity, spreading out to the wings gives the barbell **more curvature** — so on a big yield move `−D·Δy + ½·C·(Δy)²` should leave the barbell ahead, up or down. That is the textbook 'barbells are convex' free lunch. The catch the desk tests: the market makes you *pay* for convexity with a lower yield, and the barbell carries curve-reshaping (butterfly) risk the bullet doesn't.

In [1]:
R = dict(w_short=0.605, w_long=0.395, barbell_ann=2.15, bullet_ann=2.28, barbell_vol=6.41,
         bullet_vol=6.52, spread_bps=-0.054, t_nw=-0.27, sharpe_adv=-0.018, corr=0.944)
print('duration-matched barbell = %.3f*SHY + %.3f*TLT   vs   bullet = IEF'
      % (R['w_short'], R['w_long']))
print('  barbell : %+.2f%%/yr  vol %.2f%%' % (R['barbell_ann'], R['barbell_vol']))
print('  bullet  : %+.2f%%/yr  vol %.2f%%  (corr %.3f)' % (R['bullet_ann'], R['bullet_vol'], R['corr']))
print('  spread  : %+.3f bps/day  (NW t = %+.2f, Sharpe advantage %+.3f)'
      % (R['spread_bps'], R['t_nw'], R['sharpe_adv']))

duration-matched barbell = 0.605*SHY + 0.395*TLT   vs   bullet = IEF
  barbell : +2.15%/yr  vol 6.41%
  bullet  : +2.28%/yr  vol 6.52%  (corr 0.944)
  spread  : -0.054 bps/day  (NW t = -0.27, Sharpe advantage -0.018)


## 2. Is the detector any good? A live synthetic control

We plant an **under-priced** convexity in a seeded toy curve (`edge>0` ⇒ the barbell's extra convexity is a genuine net pickup) and check the spread lights up — and stays *silent* on the null (`edge=0`, convexity present but exactly paid for by a yield give-up). No network.

In [2]:
import os, sys
sys.path.insert(0, os.path.abspath('..'))
sys.path.insert(0, os.path.abspath(os.path.join('..','..','..')))
from barbell import data, strategy as st
null = st.synthetic_detect(data.synthetic_panel(edge=0.0, seed=884, n_days=1300))
planted = st.synthetic_detect(data.synthetic_panel(edge=0.6, seed=884, n_days=1800))
print('null world   : spread NW t = %+.2f  (should be ~0)' % null['t_nw'])
print('planted world: spread NW t = %+.2f  (should light up)' % planted['t_nw'])
print('convexity slope > 0 in BOTH worlds (structural): null %+.2f / planted %+.2f'
      % (null['conv_slope'], planted['conv_slope']))

null world   : spread NW t = -0.33  (should be ~0)
planted world: spread NW t = +4.30  (should light up)
convexity slope > 0 in BOTH worlds (structural): null +0.73 / planted +0.70


## 3. The honest verdict — the free lunch isn't free

On the real Treasury tape the duration-matched barbell earns **+2.15%/yr** vs the bullet's **+2.28%** at the same vol — it *under*-earns. The daily spread is **-0.054 bps** (Newey-West *t* = **-0.27**), its bootstrap CI straddles zero (**[-0.43, +0.34]**), and the excess-vs-excess Sharpe advantage is **-0.02**. Two tells seal it:

1. **The convexity is invisible in total return.** The `f²` slope is *wrong-signed* (**-0.22**) and the convexity smile is absent — the barbell does **not** systematically win when yields move most.
2. **2022 — the biggest move — went the wrong way.** In the historic selloff the barbell **-15.40%** *lost* to the bullet **-15.16%** (spread -0.38%): exactly the scenario the claim needs, and it failed.

A barbell really is more convex — but the market prices that convexity into the wings' lower yield and charges butterfly (curve-reshaping) risk the bullet avoids. **Signal: None**, **Tradability: Mirage**.